In [1]:
# Install Klygo from the v2-release branch
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

# Mining Analysis

[!["Open In Colab"](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/mining/Document_Mining_Analysis.ipynb)

Explore dataset statistical summarization, multi-criteria scoring, and comparative reporting.

## Jump to an API

- [mining.stats](#mining.stats)
- [mining.score](#mining.score)
- [mining.compare](#mining.compare)
- [mining.MiningStats](#mining.MiningStats)
- [mining.MiningScores](#mining.MiningScores)
- [mining.MiningComparison](#mining.MiningComparison)

### mining.stats

Summarize images, annotations, classes, co-occurrence, imbalance, and quality.

In [2]:

# Load a materialized YOLO dataset
from klygo import archive, datasets, files, mining

# Download and convert COCO8 into a normalized flat YOLO dataset
workspace = files.join("artifacts", "tutorials", "mining")
flat_dir = files.join(workspace, "flat_coco8")
if not files.exists(files.join(flat_dir, "data.yaml")):
    files.mkdir(workspace)
    zip_path = files.download("https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8.zip", workspace, overwrite=True, verbose=False)
    yaml_path = files.download("https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/cfg/datasets/coco8.yaml", workspace, overwrite=True, verbose=False)
    raw_dir = files.join(workspace, "raw_coco8")
    archive.extract(zip_path, raw_dir, overwrite=True, verbose=False)
    files.copy(yaml_path, files.join(raw_dir, "coco8.yaml"), overwrite=True)
    datasets.convert(
        source=raw_dir,
        target=flat_dir,
        tree={
            "images": ["coco8/images/train", "coco8/images/val"],
            "labels": ["coco8/labels/train", "coco8/labels/val"],
            "metadata": "coco8.yaml",
        },
        on_conflict="rename",
        overwrite=True,
    )

data = datasets.load(flat_dir, format="yolo", verbose=False)
print(f"Loaded dataset: {len(data)} images, {data.total_objects} boxes, classes={list(data.names.values())}")

# Compute dataset summary statistics
summary = mining.stats(data, quality=True, cooccurrence=True)
summary.show()
print("Imbalance ratio:", summary.imbalance_ratio)

Loaded dataset: 8 images, 30 boxes, classes=['bowl', 'broccoli', 'orange', 'giraffe', 'potted plant', 'vase', 'zebra', 'umbrella', 'person', 'dog', 'horse', 'elephant']
                                  MiningStats                                  
┌───────────────────┬─────────────────────────────────────────────────────────┐
│ Metric            │ Value                                                   │
├───────────────────┼─────────────────────────────────────────────────────────┤
│ images            │ 8                                                       │
│ boxes             │ 30                                                      │
│ classes           │ 12                                                      │
│ empty_images      │ 0                                                       │
│ multilabel_images │ 5                                                       │
│ images_per_class  │ {'bowl': 1, 'broccoli': 1, 'dog': 1, 'elephant': 1,     │
│                   │ 'giraffe'

### mining.score

Compute one normalized ranking value per image using quality, rarity, coverage, and box density.

In [3]:
# Score every image with weighted criteria
scores = mining.score(data, weights={"quality": 0.4, "rarity": 0.3, "coverage": 0.2, "boxes": 0.1})
scores.show()
best_idx = scores.values.index(max(scores.values))
print(f"Best image index: {best_idx} with score {scores.values[best_idx]:.4f}")

                                 MiningScores                                  
┌───────┬───────────────┬──────────────┬───────────────┬──────────────┬───────┐
│ Index │ Score         │ Quality      │ Rarity        │ Coverage     │ Boxes │
├───────┼───────────────┼──────────────┼───────────────┼──────────────┼───────┤
│ 0     │ 0.7437587174… │ 0.765646793… │ 1.0           │ 0.25         │ 0.875 │
│ 1     │ 0.5045086250… │ 0.438354895… │ 1.0           │ 0.083333333… │ 0.125 │
│ 2     │ 0.3775771704… │ 0.561502450… │ 0.3571428571… │ 0.166666666… │ 0.125 │
│ 3     │ 0.4526453295… │ 0.339946657… │ 1.0           │ 0.083333333… │ 0.0   │
│ 4     │ 0.2384634494… │ 0.374432433… │ 0.1428571428… │ 0.166666666… │ 0.125 │
│ 5     │ 0.5905079713… │ 0.684603261… │ 1.0           │ 0.083333333… │ 0.0   │
│ 6     │ 0.4257056251… │ 0.689264062… │ 0.0           │ 0.25         │ 1.0   │
│ 7     │ 0.3850781380… │ 0.647219154… │ 0.1428571428… │ 0.166666666… │ 0.5   │
└───────┴───────────────┴──────────────┴

### mining.compare

Compare the original dataset with a curated or balanced subset.

In [4]:
# Compare dataset snapshots
subset = mining.select(data, method="weighted", limit=4)
comparison = mining.compare(data, subset)
comparison.show()
print("Deltas:", comparison.changes)

                               MiningComparison                                
┌─────────┬───────────────────────────────────────────────────────────────────┐
│ Metric  │ Value                                                             │
├─────────┼───────────────────────────────────────────────────────────────────┤
│ before  │ {'images': 8, 'boxes': 30, 'classes': 12, 'empty_images': 0,      │
│         │ 'multilabel_images': 5, 'images_per_class': {'bowl': 1,           │
│         │ 'broccoli': 1, 'dog': 1, 'elephant': 1, 'giraffe': 1, 'horse': 1, │
│         │ 'orange': 1, 'person': 3, 'potted plant': 2, 'umbrella': 1,       │
│         │ 'vase': 1, 'zebra': 1}, 'boxes_per_class': {'bowl': 3,            │
│         │ 'broccoli': 1, 'dog': 1, 'elephant': 2, 'giraffe': 2, 'horse': 2, │
│         │ 'orange': 4, 'person': 10, 'potted plant': 2, 'umbrella': 1,      │
│         │ 'vase': 1, 'zebra': 1}, 'box_area': {'mean': 0.15544984346793736, │
│         │ 'median': 0.0259564923689999

### mining.MiningStats

Serializable aggregate statistics returned by `stats`, supporting `.to_dict()`, `.show()`, and `.save()`.

### mining.MiningScores

Per-image score values and component details returned by `score`.

### mining.MiningComparison

Before, after, and numeric changes returned by `compare`.

## Put it together

Analyze ground truth, rank frames by multi-criteria scores, select high-value samples, and record comparative reports.

In [5]:
summary = mining.stats(data)
summary.save("artifacts/stats.json")
scores = mining.score(data)
curated = mining.select(data, method="weighted", scores=scores, limit=4)
mining.compare(data, curated).save("artifacts/comparison.md")
print("Analysis and curation complete.")

Analysis and curation complete.


## Remember

1. `stats`, `score`, and `compare` require materialized `Detections` collections.
2. All reports are immutable and support `.show()` for terminal display as well as `.save()` for JSON/Markdown/CSV export.
3. Analysis operations never mutate inputs.